# 02 · Traer al Lakehouse las decisiones de la consola de alertas

La Fabric App guarda cada decisión de la analista en su propia SQL database, en la tabla `RevisionAlertas`. Fabric replica esa base de datos en OneLake de forma automática.

Este notebook copia esa tabla al Lakehouse como tabla Delta administrada, `revisiones`. Así la entidad **Revision** de la ontología se enlaza a una tabla del Lakehouse, como todas las demás, y puede formar parte del grafo.

**Preparación, una sola vez:**

1. En el Lakehouse `Bretema_LH`, crea un acceso directo: **Tables › Nuevo acceso directo › Microsoft OneLake**.
2. Elige la SQL database de la app. Fabric la crea al desplegar la app y aparece dentro del elemento de la app en el área de trabajo.
3. Marca la tabla `RevisionAlertas` y créalo. El acceso directo aparece en el Lakehouse con ese nombre.

Si prefieres no crear el acceso directo, pega en `RUTA_ABFS` la ruta ABFS de la tabla, que puedes copiar desde las propiedades de la tabla en la SQL database.

En la demo, ejecútalo después de registrar decisiones en la app. También puedes programarlo cada pocos minutos.

In [ ]:
# Parámetros
TABLA_ATAJO = "RevisionAlertas"  # acceso directo a la tabla de la app (en Lakehouses con esquemas: "dbo.RevisionAlertas")
RUTA_ABFS = ""                    # alternativa: ruta ABFS de la tabla en OneLake; si la rellenas, se usa en lugar del acceso directo
TABLA_DESTINO = "revisiones"

In [ ]:
from pyspark.sql import functions as F

origen = spark.read.format("delta").load(RUTA_ABFS) if RUTA_ABFS else spark.read.table(TABLA_ATAJO)

revisiones = (origen.select(
        F.col("id").cast("string").alias("revision_id"),
        F.col("alerta_id").cast("string").alias("alerta_id"),
        F.col("decision").cast("string").alias("decision"),
        F.col("bloquear_tarjeta").cast("boolean").alias("bloquear_tarjeta"),
        F.col("comentario").cast("string").alias("comentario"),
        F.col("analista_email").cast("string").alias("analista_email"),
        F.col("revisada_en").cast("timestamp").alias("revisada_en"))
    .where(F.col("revision_id").isNotNull()))

revisiones.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TABLA_DESTINO)
print(f"{spark.table(TABLA_DESTINO).count()} revisiones copiadas a {TABLA_DESTINO}")

In [ ]:
display(spark.sql(f"""
    SELECT r.revisada_en, r.alerta_id, a.tipo_alerta, r.decision, r.bloquear_tarjeta, r.analista_email, r.comentario
    FROM {TABLA_DESTINO} r LEFT JOIN alertas a ON a.alerta_id = r.alerta_id
    ORDER BY r.revisada_en DESC
"""))

**Siguiente paso:** en la ontología, actualiza el grafo (**Inicio › Administrar › Graph** y el icono de actualizar de **Última actualización**) para que el agente de datos vea las decisiones nuevas.